In [ ]:
import torch
import numpy as np
import umap
import plotly.graph_objects as go
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from sklearn.metrics.pairwise import cosine_distances

# 1. Database path (Update this to point to your new natural_gallery_...pt file)
DB_PATH = ""

# 2. Prepare the data
if 'X_3d' not in locals():
    print("Loading pre-calculated K-centroids...")
    data = torch.load(DB_PATH, map_location='cpu')
    
    # Extract centroids instead of raw embeddings based on your new generator script
    raw_centroids = data.get('centroids', data.get('embeddings'))
    if isinstance(raw_centroids, torch.Tensor):
        raw_centroids = raw_centroids.numpy()
        
    labels = data['labels']
    labels_keys = data.get('labels_keys', {})
    
    centroids = []
    class_names = []
    label_counts = {}
    
    # Treat each centroid as an independent point with a _k{X} suffix
    for i in range(len(raw_centroids)):
        label_id = int(labels[i])
        
        # Track how many times we've seen this label to assign k1, k2, k3...
        label_counts[label_id] = label_counts.get(label_id, 0) + 1
        k_idx = label_counts[label_id]
        
        base_name = labels_keys.get(label_id, {}).get('label', f"Class {label_id}")
        unique_name = f"{base_name}_k{k_idx}"
        
        class_names.append(unique_name)
        centroids.append(raw_centroids[i])
        
    centroids = np.array(centroids)
    
    # Calculate pairwise distances treating every k-centroid independently
    dist_matrix = cosine_distances(centroids)
    
    print("Calculating UMAP 3D...")
    reducer = umap.UMAP(n_components=3, n_neighbors=15, min_dist=0.3, metric='cosine', random_state=42)
    X_3d = reducer.fit_transform(centroids)
    
    df_base = pd.DataFrame({'x': X_3d[:, 0], 'y': X_3d[:, 1], 'z': X_3d[:, 2], 'Fish Name': class_names})
    print("✅ Base ready!")

# 3. UI widgets
style = {'description_width': 'initial'}
sorted_class_names = sorted(class_names)

dropdown = widgets.Dropdown(options=sorted_class_names, description='🐟 Target centroid:', style=style, layout={'width': '400px'})

search_box = widgets.Combobox(
    placeholder='Start typing fish name...',
    options=sorted_class_names,
    description='⚖️ Add to compare:',
    ensure_option=True,
    style=style,
    layout={'width': '250px'}
)

btn_add = widgets.Button(description='Add', button_style='info', layout={'width': '70px'})
btn_clear = widgets.Button(description='Clear', button_style='warning', layout={'width': '70px'})
compare_controls = widgets.HBox([search_box, btn_add, btn_clear])

selected_fishes_html = widgets.HTML(value="<b>Selected for comparison (0/5):</b> None")
slider = widgets.IntSlider(value=5, min=1, max=100, step=1, description='🔍 Top-K neighbors?', style=style, layout={'width': '400px'})
save_button = widgets.Button(description='💾 Save as HTML', button_style='success', layout={'width': '400px'})

output_ui = widgets.Output()
current_fig = None
manual_compare_list = []

def add_fish_to_compare(b):
    global manual_compare_list
    fish_name = search_box.value
    if fish_name and fish_name in class_names:
        if fish_name not in manual_compare_list and len(manual_compare_list) < 5:
            manual_compare_list.append(fish_name)
            search_box.value = ''
            update_compare_display()
            update_plot()

def clear_compare_list(b):
    global manual_compare_list
    manual_compare_list.clear()
    search_box.value = ''
    update_compare_display()
    update_plot()

def update_compare_display():
    if not manual_compare_list:
        selected_fishes_html.value = "<b>Selected for comparison (0/5):</b> None"
    else:
        fishes_str = "<br>".join([f"• {f}" for f in manual_compare_list])
        selected_fishes_html.value = f"<b>Selected for comparison ({len(manual_compare_list)}/5):</b><br>{fishes_str}"

btn_add.on_click(add_fish_to_compare)
btn_clear.on_click(clear_compare_list)

def update_plot(change=None):
    global current_fig
    target_fish = dropdown.value
    top_k = slider.value
    target_idx = class_names.index(target_fish)
    
    compare_indices = [class_names.index(f) for f in manual_compare_list if f != target_fish]
    dists = dist_matrix[target_idx]
    nearest_indices = np.argsort(dists)[:top_k + 1]
    
    table_data = []
    for rank, idx in enumerate(nearest_indices[1:], 1):
        sim_score = (1.0 - dists[idx]) * 100
        table_data.append({'Rank': rank, 'Centroid Name': class_names[idx], 'Similarity': sim_score})
    df_table = pd.DataFrame(table_data)
    
    compare_data = []
    for idx in compare_indices:
        sim_score = (1.0 - dists[idx]) * 100
        compare_data.append({'Centroid Name': class_names[idx], 'Similarity': sim_score})
    df_compare = pd.DataFrame(compare_data) if compare_data else None

    fig = go.Figure()
    highlight_indices = set(nearest_indices).union(set(compare_indices))
    bg_mask = ~np.isin(np.arange(len(class_names)), list(highlight_indices))
    
    fig.add_trace(go.Scatter3d(
        x=df_base['x'][bg_mask], y=df_base['y'][bg_mask], z=df_base['z'][bg_mask],
        mode='markers', hovertext=df_base['Fish Name'][bg_mask], hoverinfo="text",
        marker=dict(size=3, color='lightgrey', opacity=0.4), name='Other centroids'
    ))
    
    tx, ty, tz = df_base.loc[target_idx, ['x', 'y', 'z']]
    
    for i, idx in enumerate(nearest_indices[1:]):
        nx, ny, nz = df_base.loc[idx, ['x', 'y', 'z']]
        line_opacity = max(0.2, 1.0 - (i / top_k)) 
        fig.add_trace(go.Scatter3d(
            x=[tx, nx], y=[ty, ny], z=[tz, nz], mode='lines',
            line=dict(color='orange', width=2), opacity=line_opacity, hoverinfo='none', showlegend=False
        ))
        sim_score = (1.0 - dists[idx]) * 100
        fig.add_trace(go.Scatter3d(
            x=[nx], y=[ny], z=[nz], mode='markers',
            hovertext=f"{class_names[idx]}<br>Similarity: {sim_score:.1f}%", hoverinfo="text",
            marker=dict(size=7, color='orange', line=dict(width=1, color='black')), showlegend=False
        ))
        
    for idx in compare_indices:
        nx, ny, nz = df_base.loc[idx, ['x', 'y', 'z']]
        sim_score = (1.0 - dists[idx]) * 100
        fig.add_trace(go.Scatter3d(
            x=[tx, nx], y=[ty, ny], z=[tz, nz], mode='lines',
            line=dict(color='cyan', width=3, dash='dash'), hoverinfo='none', showlegend=False
        ))
        fig.add_trace(go.Scatter3d(
            x=[nx], y=[ny], z=[nz], mode='markers',
            hovertext=f"MANUAL: {class_names[idx]}<br>Similarity: {sim_score:.1f}%", hoverinfo="text",
            marker=dict(size=9, color='cyan', symbol='square', line=dict(width=1, color='black')), showlegend=False
        ))
        
    fig.add_trace(go.Scatter3d(
        x=[tx], y=[ty], z=[tz], mode='markers',
        hovertext=f"TARGET: {target_fish}", hoverinfo="text",
        marker=dict(size=12, color='red', symbol='diamond', line=dict(width=2, color='black')), name='Selected centroid'
    ))
    
    fig.update_layout(
        title=f"Connection graph for: {target_fish}",
        margin=dict(l=0, r=0, b=0, t=40),
        scene=dict(xaxis_title='', yaxis_title='', zaxis_title='', 
                   xaxis=dict(showgrid=False), yaxis=dict(showgrid=False), zaxis=dict(showgrid=False)),
        showlegend=True, legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
    )
    current_fig = fig
    
    with output_ui:
        clear_output(wait=True)
        
        if df_compare is not None:
            display(HTML("<h3>⚖️ Manual Comparison Scores:</h3>"))
            styled_compare = df_compare.style.hide(axis='index') \
                .format({'Similarity': "{:.1f}%"}) \
                .background_gradient(cmap='Blues', subset=['Similarity'])
            display(styled_compare)
            
        display(HTML(f"<h3>🔝 Top-{top_k} similar centroids:</h3>"))
        styled_table = df_table.style.hide(axis='index') \
            .format({'Similarity': "{:.1f}%"}) \
            .background_gradient(cmap='Oranges', subset=['Similarity'])
        display(styled_table)
        fig.show()

dropdown.observe(update_plot, names='value')
slider.observe(update_plot, names='value')
update_plot()

# Removed the metrics_html panel as inter/intra distances are no longer relevant
display(widgets.VBox([
    dropdown, 
    widgets.HTML("<hr>"),
    compare_controls,
    selected_fishes_html,
    widgets.HTML("<hr>"),
    slider, 
    save_button
]))
display(output_ui)